<a href="https://colab.research.google.com/github/guilherme-oamorim/TechChallenge1-FIAP_PosDataAnalytics/blob/main/TechChallenge_FIAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Olá, mundo!")
print(pd.__version__)

Olá, mundo!
2.2.2


In [ ]:
df_customers = pd.read_csv("olist_customers_dataset.csv")
df_geolocation = pd.read_csv("olist_geolocation_dataset.csv")
df_order_items = pd.read_csv("olist_order_items_dataset.csv")
df_order_payments = pd.read_csv("olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("olist_orders_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
df_sellers = pd.read_csv("olist_sellers_dataset.csv")

### Bases

Essa base será importante para fazermos cruzamentos de dados por região (por estado, cidade, ou as 5 regiões do Brasil)

In [ ]:
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Analisando as colunas dessa seguinte, vemos que _df_geolocation_ praticamente não precisa ser usada, pois a única informação que difere da _df_customers_ é a latitude e longitude: que não é usado em análise

In [ ]:
df_geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


_df_order_items_ é o único df onde possuímos o preço tanto da compra quanto do frete. **É basicamente o coração da nossa análise**, principalmente depois de relacionada com a nossa tabela principal _df_orders_

In [ ]:
df_order_items.head(100)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
95,0035246a40f520710769010f752e7507,1,8a6187b2665118d5095f99a25fd7ba7a,4a3ca9315b744ce9f8e9374361493884,2017-08-23 01:25:39,87.00,12.11
96,0035c0b07126fe9c24a325216fb96064,1,ec02a5d380128f7a188e9ce8f3ddd832,8444e55c1f13cd5c179851e5ca5ebd00,2017-12-12 01:29:45,131.90,18.54
97,0035e6b7ade84b3f5b86bd49814793df,1,71a5f1c2a5fd9889ef26b5ac22aec9c6,537eb890efff034a88679788b647c564,2018-02-27 03:31:08,19.90,14.10
98,0036757472ece3dde52fd4bfd929c90e,1,4c1bbc12438daec98a77243c2bf7a3ba,7c67e1448b00f6e969d365cea6b010ab,2018-08-08 15:10:11,136.99,66.04


Não é essencial saber quais tipos de cartão são mais utilizados ou quais produtos são mais parcelados (a tendência é simplesmente compras mais caras serem mais parceladas). Nem quais cartões são mais utilizados para cada categoria de produto. Talvez saber qual região parcela mais...? De todo modo, esse df não parece extrair além de curiosidades (e não dados relevantes)

In [ ]:
df_order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Visto que são muitos dados e exigiria uma codificação demorada, infelizmente não será possível agrupar os principais tipos de reclamação, no entanto o review score ainda é útil e é o que mais servirá para entender se atrasos de frete ou coisa desagradáveis semelhantes impactam a má avaliação dos pedidos.

In [ ]:
df_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


Junto com a _df_order_items_, é a df mais importante do projeto, as duas juntas combinam o núcleo da análise, essa tem o Customer_id, e o restante dos dados é complementado pela outra.

In [ ]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Mais adiante veremos que a _df_products_ possui 610 categorias, e é basicamente essa informação que importa nesse dataframe (saber quais categorias foram compradas através de outros dfs), o restante é descartável

In [ ]:
df_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


A _df_sellers_ serve basicamente para saber de onde veio o produto, pode influenciar nas considerações sobre frete

In [ ]:
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Exploração do dataset

In [ ]:
dfs = {
    "clientes": df_customers,
    "local": df_geolocation,
    "itens pedidos": df_order_items,
    "pagamentos":df_order_payments,
    "reviews":df_order_reviews,
    "pedidos": df_orders,
    "produtos": df_products,
    "vendedores": df_sellers,
}

In [ ]:
for nome, df in dfs.items():
    print(f"\n{'='*50}")
    print(f"DATAFRAME: {nome}")
    print(f"{'='*50}")

    print(f"\nDimensões: {df.shape}")
    print(f"\nColunas:")
    print(df.columns.tolist())

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nValores ausentes:")
    print(df.isna().sum())

    print("\nDuplicatas:")
    print(df.duplicated().sum())


DATAFRAME: clientes

Dimensões: (99441, 5)

Colunas:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Tipos de dados:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Valores ausentes:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicatas:
0

DATAFRAME: local

Dimensões: (1000163, 5)

Colunas:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Tipos de dados:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Valores ausentes:
geolocation_zip_code_prefix    0


In [ ]:
for nome, df in dfs.items():
    print(f"\nDATAFRAME: {nome} ----------> dimensões: {df.shape}")
    print(f"{df.columns.tolist()}")




DATAFRAME: clientes ----------> dimensões: (99441, 5)
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

DATAFRAME: local ----------> dimensões: (1000163, 5)
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

DATAFRAME: itens pedidos ----------> dimensões: (112650, 7)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

DATAFRAME: pagamentos ----------> dimensões: (103886, 5)
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

DATAFRAME: reviews ----------> dimensões: (99224, 7)
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

DATAFRAME: pedidos ----------> dimensões: (99441, 8)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_c

Parece que a quantidade "certa" de linhas é de 99441, visto que _pedidos_ e _clientes_, que não podem se repetir, tem essa quantidade

In [ ]:
print(f"Pedidos: {df_orders.shape[0]}")
print(f"Itens de pedido: {df_order_items.shape[0]}")
print(f"Pagamentos: {df_order_payments.shape[0]}")
print(f"Reviews: {df_order_reviews.shape[0]}")
print(f"Clientes: {df_customers.shape[0]}")

Pedidos: 99441
Itens de pedido: 112650
Pagamentos: 103886
Reviews: 99224
Clientes: 99441


## Tratamento e merge

Para ter em mente o valor inicial e garantir que não tenha dados duplicados:

In [ ]:
df_orders['order_id'].nunique()
print(f"Pedidos: {df_orders['order_id'].nunique()}")
print(f"Clientes únicos: {df_customers['customer_unique_id'].nunique()}")
print(f"Itens de pedido: {df_order_items['order_id'].nunique()}")
print(f"Pagamentos: {df_order_payments['order_id'].nunique()}")
print(f"Reviews: {df_order_reviews['order_id'].nunique()}")
print(f"Produtos: {df_products['product_id'].nunique()}")
print(f"Vendedores: {df_sellers['seller_id'].nunique()}")


Pedidos: 99441
Clientes únicos: 96096
Itens de pedido: 98666
Pagamentos: 99440
Reviews: 98673
Produtos: 32951
Vendedores: 3095


### Pedidos por mês

Dados do ano de 2016 e dos últimos 2 meses de 2018 estão inconsistentes

In [ ]:
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])

df_orders['ano_mes'] = df_orders['order_purchase_timestamp'].dt.to_period('M')

pedidos_por_mes = df_orders.groupby('ano_mes')['order_id'].nunique().reset_index()
pedidos_por_mes.columns = ['ano_mes', 'qtd_pedidos']

print(pedidos_por_mes)

    ano_mes  qtd_pedidos
0   2016-09            4
1   2016-10          324
2   2016-12            1
3   2017-01          800
4   2017-02         1780
5   2017-03         2682
6   2017-04         2404
7   2017-05         3700
8   2017-06         3245
9   2017-07         4026
10  2017-08         4331
11  2017-09         4285
12  2017-10         4631
13  2017-11         7544
14  2017-12         5673
15  2018-01         7269
16  2018-02         6728
17  2018-03         7211
18  2018-04         6939
19  2018-05         6873
20  2018-06         6167
21  2018-07         6292
22  2018-08         6512
23  2018-09           16
24  2018-10            4


O último mês de 2018 sequer termina no final

In [ ]:
print(df_orders['order_purchase_timestamp'].min())
print(df_orders['order_purchase_timestamp'].max())

2016-09-04 21:15:19
2018-10-17 17:30:18


Isso daria uma impressão errada do dado, quase como uima "falência total do e-commerce", como mostra o gráfico:

In [ ]:
pedidos_por_mes['ano_mes'] = pedidos_por_mes['ano_mes'].astype(str)

fig = px.line(pedidos_por_mes, x='ano_mes', y='qtd_pedidos',
               title='Evolução de Pedidos por Mês',
               markers=True)
fig.update_layout(xaxis_title='', yaxis_title='Nº de Pedidos')
fig.show()

No entanto, filtrando esses dados inconsistentes, a aparência é outra, inclusive de **possível retomada e ascensão**

In [ ]:
pedidos_por_mes_v2 = pedidos_por_mes[
    (pedidos_por_mes['ano_mes'] >= '2017-01') &
    (pedidos_por_mes['ano_mes'] <= '2018-08')
].reset_index(drop=True)

fig = px.line(pedidos_por_mes_v2, x='ano_mes', y='qtd_pedidos',
               title='Evolução de Pedidos por Mês',
               markers=True)
fig.update_layout(xaxis_title='', yaxis_title='Nº de Pedidos')
fig.show()

### Nova base (sem dados de 2016 e últimos 2 meses de 2018)

In [ ]:
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])

df_orders = df_orders[
    (df_orders['order_purchase_timestamp'] >= '2017-01-01') &
    (df_orders['order_purchase_timestamp'] < '2018-09-01')
].reset_index(drop=True)

print(f"Novo total de pedidos: {df_orders['order_id'].nunique()}")

Novo total de pedidos: 99092


In [ ]:
order_ids_validos = df_orders['order_id'].unique()

df_order_items = df_order_items[df_order_items['order_id'].isin(order_ids_validos)].reset_index(drop=True)
df_order_payments = df_order_payments[df_order_payments['order_id'].isin(order_ids_validos)].reset_index(drop=True)
df_order_reviews = df_order_reviews[df_order_reviews['order_id'].isin(order_ids_validos)].reset_index(drop=True)

customer_ids_validos = df_orders['customer_id'].unique()
df_customers = df_customers[df_customers['customer_id'].isin(customer_ids_validos)].reset_index(drop=True)

In [ ]:
print("Novas bases:\n")

df_orders['order_id'].nunique()
print(f"Pedidos: {df_orders['order_id'].nunique()}")
print(f"Clientes únicos: {df_customers['customer_unique_id'].nunique()}")
print(f"Itens de pedido: {df_order_items['order_id'].nunique()}")
print(f"Pagamentos: {df_order_payments['order_id'].nunique()}")
print(f"Reviews: {df_order_reviews['order_id'].nunique()}")
print(f"Produtos: {df_products['product_id'].nunique()}")
print(f"Vendedores: {df_sellers['seller_id'].nunique()}")

Novas bases:

Pedidos: 99092
Clientes únicos: 95774
Itens de pedido: 98353
Pagamentos: 99092
Reviews: 98330
Produtos: 32951
Vendedores: 3095


### Merge

In [ ]:
itens_agg = df_order_items.groupby('order_id').agg(
    qtd_itens=('order_item_id', 'count'),
    valor_produtos=('price', 'sum'),
    valor_frete=('freight_value', 'sum')
).reset_index()

pagamentos_agg = df_order_payments.groupby('order_id').agg(
    valor_total_pago=('payment_value', 'sum'),
    qtd_parcelas=('payment_installments', 'max'),
    tipo_pagamento=('payment_type', lambda x: x.mode()[0])
).reset_index()

In [ ]:
itens_agg.head()

,order_id,qtd_itens,valor_produtos,valor_frete
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14
